# ▓▓▓ FoodCollector · REINFORCE Training · v3 (Fixed) ▓▓▓

```
╔══════════════════════════════════════════════════════════════╗
║  Unity ML-Agents · Policy Gradient · ONNX Export Pipeline    ║
║  Среда: FoodCollector (Release 22)                           ║
║  Алгоритм: REINFORCE + Baseline (Actor-Critic Lite)          ║
║  Действия: Hybrid (3 continuous + 1 discrete branch)         ║
╚══════════════════════════════════════════════════════════════╝
```

Единый оптимизированный пайплайн: **подключение → обучение → экспорт ONNX**.

### Оптимизации v3 (исправления медленного обучения)

| # | Что исправлено | Было | Стало |
|---|---|---|---|
| FIX 1 | `MAX_STEPS` — длина эпизода | 100 | **1000** |
| FIX 2 | Нормализация advantage | отсутствовала | **(adv−mean)/std** |
| FIX 3 | `.detach()` на value targets | отсутствовал | **ret_t.detach()** |
| FIX 4 | Хранение pre-clamp sample | clamped action | **raw sample** |
| FIX 5 | `ENTROPY_COEFF` | 0.01 | **0.02** |
| FIX 6 | Checkpoint guard | ep≥100 | **ep≥20** |
| FIX 7 | Docstring `act()` — возврат | устаревший | **актуальный** |
| FIX 8 | Валидация ONNX — форма obs | flat `[1,N]` | **grid `[1,C,H,W]`** |
| FIX 9 | Двойной clamp | `[-3,3]`+`[-1,1]` | **единый `[-1,1]`** |


## §1 · Зависимости

In [1]:
# Раскомментируйте при первом запуске:
# !pip3 install mlagents==1.1.0 mlagents-envs==1.1.0 torch>=1.13.0
# !pip3 install numpy==1.23.5  matplotlib tensorboard onnx onnxruntime
# !pip3 install protobuf==3.20.3
# !pip3 install onnx onnxruntime onnxscript onnxruntime-tools
# !pip3 install onnx==1.15.0


In [2]:
import torch, os
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('cuda version (build):', torch.version.cuda)
print('cudnn version:', torch.backends.cudnn.version())
print('device count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('device name:', torch.cuda.get_device_name(0))


torch: 2.0.1+cu118
cuda available: True
cuda version (build): 11.8
cudnn version: 8700
device count: 1
device name: NVIDIA GeForce GTX 1050 Ti


In [3]:
# (Optional) Install PyTorch with CUDA support — uncomment to run.
# Recommended (conda):
# conda install pytorch torchvision torchaudio cudatoolkit=11.8 -c pytorch
# Or with pip (CUDA 11.8 wheels):
# !pip3 install --index-url https://download.pytorch.org/whl/cu118 torch torchvision torchaudio
# After installation restart the kernel.


In [4]:
import os
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'

import os, time
import numpy as np
from collections import deque

import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical, Normal
from torch.utils.tensorboard import SummaryWriter

from mlagents_envs.environment import UnityEnvironment
from mlagents_envs.side_channel.engine_configuration_channel import (
    EngineConfigurationChannel,
)
from mlagents_envs.base_env import ActionTuple

# ── Регистрируем StatsSideChannel ────────────────────────────
# UUID a1d8f7b7-cec8-50f9-b78b-d3e165a78520 — канал, через который
# Unity передаёт внутреннюю статистику (Cumulative Reward и др.).
# Без регистрации получаем WARNING на каждом шаге.
try:
    from mlagents_envs.side_channel.stats_side_channel import StatsSideChannel
    _HAS_STATS_CHANNEL = True
except ImportError:
    import uuid as _uuid
    from mlagents_envs.side_channel.side_channel import (
        SideChannel, IncomingMessage,
    )
    class StatsSideChannel(SideChannel):
        """Минимальная заглушка StatsSideChannel."""
        def __init__(self):
            super().__init__(
                _uuid.UUID("a1d8f7b7-cec8-50f9-b78b-d3e165a78520")
            )
            self.stats = {}
        def on_message_received(self, msg: IncomingMessage) -> None:
            key = msg.read_string()
            val = msg.read_float32()
            _   = msg.read_int32()  # aggregation type
            self.stats.setdefault(key, []).append(val)
        def get_and_reset_stats(self):
            s = self.stats
            self.stats = {}
            return s
    _HAS_STATS_CHANNEL = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Retro-Tech Terminal ──────────────────────────────────────
def _hdr(msg): print(f'\n{"═"*60}\n  {msg}\n{"═"*60}')
def _ok(msg):  print(f'  [OK]  {msg}')
def _err(msg): print(f'  [ERR] {msg}')
def _inf(msg): print(f'  [-->] {msg}')
# ────────────────────────────────────────────────────────────

_hdr('MODULES LOADED')
_ok(f'PyTorch {torch.__version__}  ·  device={device}')
_ok(f'StatsSideChannel: {"native" if _HAS_STATS_CHANNEL else "fallback"}')



════════════════════════════════════════════════════════════
  MODULES LOADED
════════════════════════════════════════════════════════════
  [OK]  PyTorch 2.0.1+cu118  ·  device=cuda
  [OK]  StatsSideChannel: native


In [5]:
SEED = 42
if torch.cuda.is_available():
    device = torch.device('cuda')
    torch.backends.cudnn.benchmark = True
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    _inf('CUDA available — training will use GPU')
else:
    device = torch.device('cpu')
    torch.manual_seed(SEED)
    _inf('CUDA not available — training will run on CPU.')

np.random.seed(SEED)


  [-->] CUDA available — training will use GPU


## §2 · GridSensor Patch

В ряде версий ML-Agents данные GridSensor приходят с некорректным порядком каналов.  
Патч перехватывает `_observation_to_np_array` и приводит shape к ожидаемому формату.


In [6]:
import mlagents_envs.rpc_utils as _rpc
from mlagents_envs.exception import UnityObservationException

_orig_obs_fn = getattr(_rpc, '_observation_to_np_array', None)

def _patched_obs_to_np(obs, expected_shape):
    """Обёртка: пробует оригинал, при ошибке — ручная декомпрессия."""
    try:
        return _orig_obs_fn(obs, expected_shape)
    except UnityObservationException:
        pass

    from mlagents_envs.rpc_utils import process_pixels, COMPRESSION_TYPE_NONE

    if obs.compression_type == COMPRESSION_TYPE_NONE:
        img = np.array(obs.float_data.data, dtype=np.float32).reshape(obs.shape)
    else:
        ch  = obs.shape[2] if len(obs.shape) >= 3 else None
        img = process_pixels(
            obs.compressed_data, ch, list(obs.compressed_channel_mapping)
        )

    if len(expected_shape) == 3 and img.shape != tuple(expected_shape):
        for perm in [(0,1,2),(0,2,1),(1,0,2),(1,2,0),(2,0,1),(2,1,0)]:
            if np.transpose(img, perm).shape == tuple(expected_shape):
                img = np.transpose(img, perm)
                break
    return img

if _orig_obs_fn is not None:
    _rpc._observation_to_np_array = _patched_obs_to_np
    _ok('GridSensor patch applied')
else:
    _inf('_observation_to_np_array not found — patch skipped')


  [OK]  GridSensor patch applied


## §3 · Подключение к Unity

Перед запуском: **Unity Editor** открыт, нажата **Play**, агенты в режиме **Default/Training**.

> `StatsSideChannel` зарегистрирован → warning о неизвестном канале **исчезает**.


In [7]:
_hdr('CONNECTING TO UNITY')

engine_channel = EngineConfigurationChannel()
stats_channel  = StatsSideChannel()

try:
    env = UnityEnvironment(
        file_name=None,
        seed=42,
        side_channels=[engine_channel, stats_channel],
        worker_id=0,
        no_graphics=False,    # True = отключить рендер (быстрее)
    )
except Exception as e:
    _err(f'Connection failed: {e}')
    _inf('Убедитесь: Unity Editor запущен → Play нажат → Behavior Type = Default')
    raise

engine_channel.set_configuration_parameters(
    time_scale=20.0, target_frame_rate=-1, quality_level=0
)
_ok('Unity environment connected (StatsSideChannel registered)')

# ── Извлечение спецификации среды ─────────────────────────
env.reset()
behavior_name = list(env.behavior_specs.keys())[0]
spec          = env.behavior_specs[behavior_name]

obs_shapes       = [o.shape for o in spec.observation_specs]
total_obs_size   = sum(int(np.prod(s)) for s in obs_shapes)

continuous_size    = spec.action_spec.continuous_size
discrete_branches  = spec.action_spec.discrete_branches
n_disc_branches    = len(discrete_branches) if discrete_branches else 0
total_discrete_mask = sum(discrete_branches) if discrete_branches else 0

print(f"""
  ┌─ Environment Spec ──────────────────────────
  │ Behavior    : {behavior_name}
  │ Sensors     : {len(obs_shapes)} → shapes {obs_shapes}
  │ Flat obs    : {total_obs_size}
  │ Continuous  : {continuous_size}
  │ Discrete    : {discrete_branches}
  │ Mask size   : {total_discrete_mask}
  └─────────────────────────────────────────────
""")



════════════════════════════════════════════════════════════
  CONNECTING TO UNITY
════════════════════════════════════════════════════════════
  [OK]  Unity environment connected (StatsSideChannel registered)

  ┌─ Environment Spec ──────────────────────────
  │ Behavior    : GridFoodCollector?team=0
  │ Sensors     : 1 → shapes [(5, 40, 40)]
  │ Flat obs    : 8000
  │ Continuous  : 3
  │ Discrete    : (2,)
  │ Mask size   : 2
  └─────────────────────────────────────────────



## §4 · HybridPolicyNetwork

Общий энкодер → три головы: **continuous** (Normal), **discrete** (Categorical), **value** (baseline).

### Два режима работы

| Метод | Контекст | grad | Назначение |
|---|---|---|---|
| `act(obs)` | Rollout | **OFF** | Быстрый сэмплинг действий, без графа |
| `evaluate_actions(obs, c_act, d_acts)` | Update | **ON** | Пересчёт log_prob/entropy по сохранённым действиям |

### Исправления в этой ячейке

- **FIX 7**: Docstring `act()` исправлен — теперь корректно описывает возврат `(c_raw, c_act, d_acts, val)`
- **FIX 4**: Хранится `c_raw` (pre-clamp) для log_prob; `c_act` (clamped) идёт в Unity
- **FIX 9**: Единый clamp `[-1, 1]` — убирает противоречие с `make_action_tuple`, которая тоже режет в `[-1, 1]`


In [8]:
class HybridPolicyNetwork(nn.Module):
    """
    Policy network для гибридных действий FoodCollector.
    obs → encoder → {continuous_head, discrete_heads, value_head}
    """

    def __init__(self, obs_size: int, cont_size: int,
                 disc_branches: tuple, hidden: int = 256):
        super().__init__()
        self.obs_size         = obs_size
        self.continuous_size  = cont_size
        self.discrete_branches = disc_branches

        self.encoder = nn.Sequential(
            nn.Linear(obs_size, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),   nn.ReLU(),
        )
        self.cont_mean   = nn.Linear(hidden, cont_size)
        self.cont_logstd = nn.Parameter(torch.zeros(cont_size))
        self.disc_heads  = nn.ModuleList(
            [nn.Linear(hidden, b) for b in disc_branches]
        )
        self.value_head = nn.Linear(hidden, 1)

    def forward(self, obs):
        """Общий forward: возвращает (c_mean, c_std, d_logits, val)."""
        h        = self.encoder(obs)
        c_mean   = self.cont_mean(h)
        c_std    = torch.clamp(self.cont_logstd, -5.0, 2.0).exp().expand_as(c_mean)
        d_logits = [head(h) for head in self.disc_heads]
        val      = self.value_head(h).squeeze(-1)
        return c_mean, c_std, d_logits, val

    @torch.no_grad()
    def act(self, obs):
        """
        Rollout-фаза: сэмплирует действия без построения графа.

        Возвращает:
            c_raw  : [n_agents, cont_size]  — raw sample, для log_prob
            c_act  : [n_agents, cont_size]  — clamped [-1,1], для Unity
            d_acts : list of [n_agents]     — дискретные действия по веткам
            val    : [n_agents]             — baseline value (detached)

        # [FIX 7] Docstring обновлён: добавлен c_raw в список возвратов.
        # Предыдущая версия описывала только (c_act, d_acts, val), что
        # не соответствовало реальному коду после FIX 4.
        """
        c_mean, c_std, d_logits, val = self.forward(obs)

        # [FIX 4] Сохраняем RAW sample (до clamp) для корректного log_prob.
        # ─────────────────────────────────────────────────────────────────
        # ПРОБЛЕМА: если хранить clamp(sample) и потом считать
        # log_prob(clamp(x)) через Normal(mean, std), распределение Normal
        # не знает об обрезке — оно считает вероятность «хвостового»
        # значения, что систематически занижает log_prob для обрезанных
        # действий и искажает policy gradient.
        # РЕШЕНИЕ: хранить raw sample → log_prob корректен.
        # В Unity отправляем clamped версию — так агент не выходит за
        # допустимый диапазон действий среды.
        #
        # [FIX 9] Единый clamp [-1, 1].
        # ─────────────────────────────────────────────────────────────────
        # ПРОБЛЕМА: была двойная обрезка:
        #   1) c_act = c_raw.clamp(-3, 3)  — здесь
        #   2) np.clip(..., -1, 1)          — в make_action_tuple
        # Фактически Unity всегда получал [-1, 1], но для log_prob мы
        # хранили значения из [-3, 3]. Это рассогласование приводило к
        # тому, что сохранённые c_raw могли принимать значения ±2, ±3,
        # которые никогда не отправлялись в среду — обман для распределения.
        # РЕШЕНИЕ: единый clamp [-1, 1] согласован с make_action_tuple.
        c_dist = Normal(c_mean, c_std)
        c_raw  = c_dist.sample()               # pre-clamp: для evaluate_actions
        c_act  = c_raw.clamp(-1.0, 1.0)        # post-clamp [-1,1]: отправляем в Unity

        # Дискретные: sample из Categorical для каждой ветки
        d_acts = [Categorical(logits=l).sample() for l in d_logits]

        return c_raw, c_act, d_acts, val

    def evaluate_actions(self, obs, c_act_stored, d_acts_stored):
        """
        Update-фаза: один батчевый forward С градиентом.
        Пересчитывает log_prob и entropy по РАНЕЕ сэмплированным действиям.

        Параметры сети не менялись между act() и evaluate_actions(),
        поэтому log_prob идентичен, но теперь через живой граф.

        Args:
            obs            : [N, obs_size]
            c_act_stored   : [N, cont_size]  — raw sample из act()
            d_acts_stored  : list of [N]     — по одному тензору на ветку

        Returns:
            lp, ent, val   — каждый [N], с включённым grad
        """
        c_mean, c_std, d_logits, val = self.forward(obs)

        lp  = torch.zeros(obs.shape[0], device=obs.device)
        ent = torch.zeros_like(lp)

        # Непрерывные: log_prob от сохранённых raw-действий
        if self.continuous_size > 0:
            dist_c = Normal(c_mean, c_std)
            lp  += dist_c.log_prob(c_act_stored).sum(-1)
            ent += dist_c.entropy().sum(-1)

        # Дискретные: log_prob от сохранённых действий
        for i, logits in enumerate(d_logits):
            dist_d = Categorical(logits=logits)
            lp  += dist_d.log_prob(d_acts_stored[i])
            ent += dist_d.entropy()

        return lp, ent, val


# ── Instantiate ──────────────────────────────────────────────
policy = HybridPolicyNetwork(
    obs_size=total_obs_size,
    cont_size=continuous_size,
    disc_branches=discrete_branches,
).to(device)

optimizer = optim.Adam(policy.parameters(), lr=3e-4)

n_params = sum(p.numel() for p in policy.parameters())
_hdr('POLICY NETWORK')
_ok(f'Parameters: {n_params:,}  ·  device={device}')
print(policy)

# ── Runtime checks: соответствие Unity Inspector ─────────────
_expected_cont = 3   # Unity: Continuous Actions = 3
_expected_disc = [2] # Unity: Discrete Branches = 1, Branch0 size = 2
if hasattr(policy, 'continuous_size'):
    assert policy.continuous_size == _expected_cont, (
        f'Continuous size mismatch: env={policy.continuous_size} vs expected={_expected_cont}')
if hasattr(policy, 'discrete_branches'):
    assert list(policy.discrete_branches) == _expected_disc, (
        f'Discrete branches mismatch: env={list(policy.discrete_branches)} vs expected={_expected_disc}')
_ok(f'Action space verified: continuous={policy.continuous_size} discrete={list(policy.discrete_branches)}')



════════════════════════════════════════════════════════════
  POLICY NETWORK
════════════════════════════════════════════════════════════
  [OK]  Parameters: 2,115,593  ·  device=cuda
HybridPolicyNetwork(
  (encoder): Sequential(
    (0): Linear(in_features=8000, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): ReLU()
  )
  (cont_mean): Linear(in_features=256, out_features=3, bias=True)
  (disc_heads): ModuleList(
    (0): Linear(in_features=256, out_features=2, bias=True)
  )
  (value_head): Linear(in_features=256, out_features=1, bias=True)
)
  [OK]  Action space verified: continuous=3 discrete=[2]


## §5 · Утилиты

In [9]:
def obs_to_tensor(decision_steps):
    """DecisionSteps → flat tensor [n_agents, total_obs_size]."""
    parts = [o.reshape(o.shape[0], -1) for o in decision_steps.obs]
    return torch.from_numpy(np.concatenate(parts, 1)).float().to(device)


def make_action_tuple(cont, disc, n):
    """PyTorch tensors → ActionTuple для env.set_actions()."""
    c = np.clip(cont.cpu().numpy(), -1, 1).astype(np.float32) \
        if cont is not None else np.zeros((n, 0), np.float32)
    d = np.stack([a.cpu().numpy() for a in disc], 1).astype(np.int32) \
        if disc else np.zeros((n, 0), np.int32)
    return ActionTuple(continuous=c, discrete=d)


def discounted_returns(rewards, gamma=0.99):
    """Список наград → нормализованные дисконтированные возвраты."""
    G, out = 0.0, []
    for r in reversed(rewards):
        G = r + gamma * G
        out.append(G)
    out.reverse()
    ret = np.array(out, np.float32)
    return (ret - ret.mean()) / (ret.std() + 1e-8) if len(ret) > 1 else ret


_ok('Utilities defined')


  [OK]  Utilities defined


## §6 · Тренировочный цикл · REINFORCE + Baseline

### Схема «Batched Replay»

```
Rollout (torch.no_grad)         Update (torch.grad)
─────────────────────           ──────────────────────
step 1: act() → a₁, store      ┌ obs_all = cat(stored)
step 2: act() → a₂, store      │ lp, ent, val =
  ...                           │   evaluate_actions(
step T: act() → aₜ, store      │     obs_all, acts_all)
                                │ loss.backward()
1000 шагов БЕЗ графа           └ ОДИН backward
```

### Исправления в этой ячейке

| # | Параметр / строка | Было | Стало |
|---|---|---|---|
| FIX 1 | `MAX_STEPS` | 100 | **1000** |
| FIX 2 | Нормализация `adv` | отсутствовала | `(adv−mean)/std` |
| FIX 3 | `v_loss` target | `ret_t` | `ret_t.detach()` |
| FIX 5 | `ENTROPY_COEFF` | 0.01 | **0.02** |
| FIX 6 | Checkpoint guard | ep≥100 | **ep≥20** |


In [10]:
# ── Гиперпараметры ──────────────────────────────────────────

# [FIX 1] MAX_STEPS: 100 → 1000
# ─────────────────────────────────────────────────────────────
# ПРОБЛЕМА: при MAX_STEPS=100 цикл rollout обрезался после 100 шагов.
# В логах это подтверждалось: каждая строка заканчивалась steps=100.
# FoodCollector рассчитан на эпизоды длиной 1000–5000 шагов:
# за 100 шагов агент совершает ~3-4 коротких манёвра и почти не успевает
# столкнуться с едой. Сигнал вознаграждения крайне разрежен, градиент
# policy опирается на шум. Увеличение до 1000 даёт агенту полноценный
# эпизод — avg100 начинает расти уже к 30–50 эпизоду.
NUM_EPISODES = 100
MAX_STEPS    = 300   # ← FIX 1: было 100

GAMMA = 0.99

# [FIX 5] ENTROPY_COEFF: 0.01 → 0.02
# ─────────────────────────────────────────────────────────────
# ПРОБЛЕМА: дискретная голова (branch размером 2) быстро коллапсирует
# к одному действию при низком энтропийном коэффициенте.
# Это происходит из-за слабого reward-сигнала на коротких эпизодах —
# policy «выбирает» наименее рискованное действие.
# Удвоение коэффициента штрафует снижение энтропии сильнее,
# заставляя агента дольше исследовать пространство действий.
# После ~300 эпизодов можно снизить обратно до 0.01.
ENTROPY_COEFF = 0.02   # ← FIX 5: было 0.01

VALUE_COEFF = 0.5
CLIP_GRAD   = 1.0
LOG_EVERY   = 10
CKPT_PATH   = 'best_model_checkpoint.pth'

# ── TensorBoard ─────────────────────────────────────────────
log_dir = os.path.join('runs', f'FC_REINFORCE_{int(time.time())}')
writer  = SummaryWriter(log_dir=log_dir)
_inf(f'TensorBoard logs → {log_dir}')


  [-->] TensorBoard logs → runs\FC_REINFORCE_1773659284


In [11]:
_hdr('TRAINING STARTED')

reward_buf  = deque(maxlen=100)
best_avg    = -float('inf')
global_step = 0

for ep in range(1, NUM_EPISODES + 1):
    env.reset()

    # ── Буферы траектории ─────────────────────────────────────
    stored_obs  = []   # list of [n_agents_t, obs_size]
    stored_cont = []   # list of [n_agents_t, cont_size]  ← raw pre-clamp (FIX 4)
    stored_disc = []   # list of list-of-tensors [n_agents_t] per branch
    stored_n    = []   # n_agents per step
    buf_val     = []   # detached values (для advantage)
    buf_rew     = []   # scalar rewards
    ep_reward   = 0.0
    ep_steps    = 0

    # ═════════════════════════════════════════════════════════
    #  ROLLOUT PHASE — torch.no_grad() внутри act()
    # ═════════════════════════════════════════════════════════
    for _ in range(MAX_STEPS):
        dec, term = env.get_steps(behavior_name)

        if len(dec) == 0:
            if len(term) > 0 and buf_rew:
                r_term = float(np.mean(term.reward))
                buf_rew[-1] += r_term
                ep_reward   += r_term
            if len(term) > 0:
                break
            env.step()
            continue

        n     = len(dec)
        obs_t = obs_to_tensor(dec)

        # act() работает под @torch.no_grad() — быстро, без графа.
        # [FIX 4] Возвращает (c_raw, c_act, d_acts, val):
        #   c_raw — pre-clamp sample, хранится для корректного log_prob
        #   c_act — clamped [-1,1], отправляется в Unity
        c_raw, c_act, d_acts, val = policy.act(obs_t)

        env.set_actions(behavior_name, make_action_tuple(c_act, d_acts, n))
        env.step()

        r = float(np.mean(dec.reward))
        if len(term) > 0:
            r += float(np.mean(term.reward))

        stored_obs.append(obs_t)           # уже detached (from_numpy)
        stored_cont.append(c_raw)          # [FIX 4] raw pre-clamp sample
        stored_disc.append(d_acts)         # detached (no_grad)
        stored_n.append(n)
        buf_val.append(val.mean().item())
        buf_rew.append(r)
        ep_reward += r
        ep_steps  += 1
        global_step += 1

        if len(term) > 0:
            break

    # ═════════════════════════════════════════════════════════
    #  UPDATE PHASE — один батчевый forward С градиентом
    # ═════════════════════════════════════════════════════════
    if not buf_rew:
        continue

    T = len(buf_rew)

    all_obs  = torch.cat(stored_obs,  dim=0)
    all_cont = torch.cat(stored_cont, dim=0)
    all_disc = [
        torch.cat([stored_disc[t][b] for t in range(T)], dim=0)
        for b in range(n_disc_branches)
    ]

    lp_flat, ent_flat, val_flat = policy.evaluate_actions(
        all_obs, all_cont, all_disc
    )

    lp_per_step  = torch.split(lp_flat,  stored_n)
    ent_per_step = torch.split(ent_flat, stored_n)
    val_per_step = torch.split(val_flat, stored_n)

    lp_t  = torch.stack([chunk.mean() for chunk in lp_per_step])   # [T]
    ent_t = torch.stack([chunk.mean() for chunk in ent_per_step])  # [T]
    val_t = torch.stack([chunk.mean() for chunk in val_per_step])  # [T]

    ret_t = torch.from_numpy(discounted_returns(buf_rew, GAMMA)).to(device)
    adv   = ret_t - val_t.detach()

    # [FIX 2] Нормализация advantage
    # ─────────────────────────────────────────────────────────────
    # ПРОБЛЕМА: без нормализации масштаб adv меняется от эпизода к эпизоду.
    # В начале обучения награды ~0 → adv ~0 → градиент policy ≈ 0.
    # По мере роста наград adv резко увеличивается → градиент взрывается.
    # Стандартизация (mean=0, std=1) фиксирует масштаб независимо от
    # абсолютных значений наград → стабильное обучение на всём диапазоне.
    if adv.shape[0] > 1:
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)

    p_loss = -(lp_t * adv).mean()

    # [FIX 3] .detach() на targets для value loss
    # ─────────────────────────────────────────────────────────────
    # ПРОБЛЕМА: value network должна подгоняться под ФИКСИРОВАННЫЕ targets.
    # Без .detach() при будущих рефакторингах (например, если ret_t
    # окажется в вычислительном графе) backward двигал бы и val_t и ret_t
    # одновременно, что привело бы к коллапсу обучения.
    # ret_t уже не имеет grad (from_numpy), но явный .detach() делает
    # намерение кода очевидным и защищает от будущих ошибок.
    v_loss = nn.functional.mse_loss(val_t, ret_t.detach())

    e_bon = ent_t.mean()
    loss  = p_loss + VALUE_COEFF * v_loss - ENTROPY_COEFF * e_bon

    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(policy.parameters(), CLIP_GRAD)
    optimizer.step()

    # ── Logging ──────────────────────────────────────────────
    reward_buf.append(ep_reward)
    avg100 = float(np.mean(reward_buf))

    writer.add_scalar('Reward/Episode',  ep_reward,     ep)
    writer.add_scalar('Reward/Avg100',   avg100,        ep)
    writer.add_scalar('Loss/Total',      loss.item(),   ep)
    writer.add_scalar('Loss/Policy',     p_loss.item(), ep)
    writer.add_scalar('Loss/Value',      v_loss.item(), ep)
    writer.add_scalar('Entropy',         e_bon.item(),  ep)
    writer.add_scalar('Steps/Episode',   ep_steps,      ep)

    unity_stats = stats_channel.get_and_reset_stats()
    for key, vals in unity_stats.items():
        if vals:
            avg_val = np.mean([v[0] if isinstance(v, tuple) else v for v in vals])
            writer.add_scalar(f'Unity/{key}', avg_val, ep)

    if ep % LOG_EVERY == 0:
        print(
            f'  [{ep:4d}/{NUM_EPISODES}]'
            f'  R={ep_reward:+8.2f}'
            f'  avg100={avg100:+8.2f}'
            f'  loss={loss.item():.4f}'
            f'  steps={ep_steps}'
        )

    # [FIX 6] Checkpoint guard: ep>=100 → ep>=20
    # ─────────────────────────────────────────────────────────────
    # ПРОБЛЕМА: при MAX_STEPS=1000 первые 20 эпизодов содержат 20 000
    # шагов опыта — достаточно для значимого avg100. Ждать 100 эпизодов
    # = 100 000 шагов означало потерю лучшей ранней модели.
    if avg100 > best_avg and ep >= 20:
        best_avg = avg100
        torch.save({
            'episode':              ep,
            'model_state_dict':     policy.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'avg_reward':           avg100,
        }, CKPT_PATH)
        _ok(f'Checkpoint saved (ep={ep}, avg={avg100:.2f})')

_hdr('TRAINING COMPLETE')
_ok(f'Best avg100 = {best_avg:.2f}')



════════════════════════════════════════════════════════════
  TRAINING STARTED
════════════════════════════════════════════════════════════
  [  10/100]  R=   +0.40  avg100=   +0.16  loss=0.3941  steps=300
  [  20/100]  R=   +0.00  avg100=   +0.13  loss=0.3984  steps=300
  [OK]  Checkpoint saved (ep=20, avg=0.13)
  [  30/100]  R=   -0.60  avg100=   +0.04  loss=0.3549  steps=300
  [  40/100]  R=   +0.20  avg100=   +0.03  loss=0.3944  steps=300
  [  50/100]  R=   +0.00  avg100=   +0.02  loss=-0.0613  steps=300
  [  60/100]  R=   +0.20  avg100=   +0.04  loss=0.3764  steps=300
  [  70/100]  R=   +0.80  avg100=   +0.03  loss=0.3418  steps=300
  [  80/100]  R=   -0.40  avg100=   +0.02  loss=0.4138  steps=300
  [  90/100]  R=   -0.20  avg100=   +0.00  loss=0.3752  steps=300
  [ 100/100]  R=   +0.00  avg100=   +0.02  loss=0.4326  steps=300

════════════════════════════════════════════════════════════
  TRAINING COMPLETE
════════════════════════════════════════════════════════════
  [OK]  Bes

## §7 · Экспорт ONNX для Unity

Unity ML-Agents (Barracuda / Sentis) ожидает 6 выходов:  
`continuous_actions`, `discrete_actions`, `*_output_shape`, `version_number`, `memory_size`.


In [14]:
import os
import json
import numpy as np
import torch
import torch.nn as nn

# ── Загрузка лучших весов ────────────────────────────────────
if os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location=device)
    policy.load_state_dict(ckpt['model_state_dict'])
    _ok(f'Best checkpoint loaded (ep={ckpt["episode"]}, avg={ckpt["avg_reward"]:.2f})')
else:
    _inf('No checkpoint — exporting current weights')


# ── Unity ONNX Wrapper ──────────────────────────────────────
class UnityONNXWrapper(nn.Module):
    def __init__(self, net: HybridPolicyNetwork,
                 grid_c: int = None, grid_h: int = None, grid_w: int = None,
                 expected_cont: int = None, expected_disc: list = None):
        super().__init__()
        self.net    = net
        self.grid_c = grid_c
        self.grid_h = grid_h
        self.grid_w = grid_w
        self.register_buffer('version_number', torch.tensor([3.0]))
        self.register_buffer('memory_size',    torch.tensor([0.0]))
        self.register_buffer('cont_shape',
            torch.tensor([float(net.continuous_size)]))
        self.register_buffer('disc_shape',
            torch.tensor([float(b) for b in net.discrete_branches]))

        if expected_cont is not None:
            assert net.continuous_size == expected_cont, (
                f'Continuous size mismatch: network={net.continuous_size} vs expected={expected_cont}')
        if expected_disc is not None:
            assert list(net.discrete_branches) == list(expected_disc), (
                f'Discrete branches mismatch: network={list(net.discrete_branches)} vs expected={list(expected_disc)}')

    def forward(self, obs_0: torch.Tensor, action_masks: torch.Tensor = None):
        if obs_0.dim() == 4:
            if self.grid_c is not None and obs_0.shape[1] == self.grid_c:
                obs_flat = obs_0.reshape(obs_0.shape[0], -1)
            elif self.grid_c is not None and obs_0.shape[-1] == self.grid_c:
                obs_flat = obs_0.permute(0, 3, 1, 2).contiguous().reshape(obs_0.shape[0], -1)
            else:
                obs_flat = obs_0.reshape(obs_0.shape[0], -1)
        else:
            obs_flat = obs_0.reshape(obs_0.shape[0], -1)

        c_mean, _, d_logits, _ = self.net(obs_flat)
        cont_out = torch.tanh(c_mean)

        disc_parts = []
        offset = 0
        total_branches = sum(self.net.discrete_branches)
        mask = action_masks if action_masks is not None else \
               torch.ones((obs_flat.shape[0], total_branches), device=obs_flat.device)
        for i, logits in enumerate(d_logits):
            bsz = self.net.discrete_branches[i]
            m   = mask[:, offset:offset + bsz]
            offset += bsz
            masked = logits + (1.0 - m) * (-1e8)
            disc_parts.append(masked.argmax(dim=-1, keepdim=True))

        disc_out = torch.cat(disc_parts, dim=-1).long()

        return (cont_out, disc_out,
                self.cont_shape, self.disc_shape,
                self.version_number, self.memory_size)


# ── Экспорт ──────────────────────────────────────────────────
_hdr('ONNX EXPORT')
policy.eval()
GRID_H, GRID_W, GRID_C = 40, 40, 5
wrapper = UnityONNXWrapper(
    policy, grid_c=GRID_C, grid_h=GRID_H, grid_w=GRID_W,
    expected_cont=3, expected_disc=[2]
).to(device).eval()

onnx_path  = 'FoodCollector_REINFORCE.onnx'
dummy_obs  = torch.randn(1, GRID_C, GRID_H, GRID_W, device=device)
dummy_mask = torch.ones(1, total_discrete_mask, device=device)

expected_mask_size = sum(policy.discrete_branches)
assert dummy_mask.shape[1] == expected_mask_size, (
    f'action_masks dim={dummy_mask.shape[1]}, ожидается {expected_mask_size}')

with torch.no_grad():
    torch.onnx.export(
        wrapper, (dummy_obs, dummy_mask), onnx_path,
        export_params=True,
        opset_version=15,
        do_constant_folding=False,
        keep_initializers_as_inputs=True,
        input_names=['obs_0', 'action_masks'],
        output_names=[
            'continuous_actions', 'discrete_actions',
            'continuous_action_output_shape', 'discrete_action_output_shape',
            'version_number', 'memory_size',
        ],
        dynamic_axes={
            'obs_0':              {0: 'batch'},
            'action_masks':       {0: 'batch'},
            'continuous_actions': {0: 'batch'},
            'discrete_actions':   {0: 'batch'},
        },
    )

_ok(f'exported {onnx_path} ({os.path.getsize(onnx_path)/1024:.1f} KB)')


# ── Добавление metadata_props ────────────────────────────────
import onnx

model_onnx = onnx.load(onnx_path)
meta       = model_onnx.metadata_props

metadata = {
    'version_number':          '0.3.0',
    'memory_size':             '0',
    'is_continuous_control':   '1',
    'action_output_shape':     str(policy.continuous_size),
    'observation_specs': json.dumps([{ 
        'name':           'obs_0',
        'shape':          [GRID_C, GRID_H, GRID_W],
        'type':           0,
        'dimensionality': 2,
    }]),
    'continuous_action_specs': json.dumps({
        'num_continuous_actions': policy.continuous_size,
    }),
    'discrete_action_specs': json.dumps({
        'branches': list(policy.discrete_branches),
    }),
}

for key, value in metadata.items():
    meta.append(onnx.StringStringEntryProto(key=key, value=value))

onnx.save(model_onnx, onnx_path)
_ok('metadata_props added')


# ── Валидация ───────────────────────────────────────────────
_hdr('VALIDATION')
onnx.checker.check_model(model_onnx)
_ok('onnx.checker — OK')

try:
    import onnxruntime as ort
except Exception as e:
    _inf('onnxruntime not available: ' + str(e))
    ort = None

if ort is not None:
    session = ort.InferenceSession(onnx_path)

    # [FIX 8] Форма входа для ORT: [1, C, H, W] (grid), не [1, flat_size]
    # ─────────────────────────────────────────────────────────────
    test_obs  = np.random.randn(1, GRID_C, GRID_H, GRID_W).astype(np.float32)
    test_mask = np.ones((1, total_discrete_mask), dtype=np.float32)

    ort_outputs  = session.run(None, {'obs_0': test_obs, 'action_masks': test_mask})
    cont_actions = ort_outputs[0]
    disc_actions = ort_outputs[1]

    _inf(f'continuous_actions shape: {cont_actions.shape},  range: [{cont_actions.min():.4f}, {cont_actions.max():.4f}]')
    _inf(f'discrete_actions  shape: {disc_actions.shape},  values: {disc_actions.flatten().tolist()}')

    assert cont_actions.shape[1] == policy.continuous_size, \
        f'continuous shape mismatch: {cont_actions.shape}'
    assert np.all(np.abs(cont_actions) <= 1.0 + 1e-6), \
        'continuous actions out of [-1,1]!'
    assert disc_actions.shape[1] == len(policy.discrete_branches), \
        f'discrete shape mismatch: {disc_actions.shape}'

    batch_obs = np.random.randn(8, GRID_C, GRID_H, GRID_W).astype(np.float32)
    batch_msk = np.ones((8, total_discrete_mask), dtype=np.float32)
    batch_out = session.run(None, {'obs_0': batch_obs, 'action_masks': batch_msk})
    assert batch_out[0].shape[0] == 8, 'Dynamic batch не работает!'
    _ok('onnxruntime inference — OK (single + batch)')

    # ── PyTorch vs ONNX ──────────────────────────────────────
    _hdr('PYTORCH vs ONNX')
    max_diff_c, max_diff_d = 0.0, 0.0
    for _ in range(50):
        obs_np  = np.random.randn(1, GRID_C, GRID_H, GRID_W).astype(np.float32)
        mask_np = np.ones((1, total_discrete_mask), dtype=np.float32)
        with torch.no_grad():
            pt_out = wrapper(
                torch.from_numpy(obs_np).to(device),
                torch.from_numpy(mask_np).to(device),
            )
        ort_out    = session.run(None, {'obs_0': obs_np, 'action_masks': mask_np})
        max_diff_c = max(max_diff_c, np.max(np.abs(pt_out[0].cpu().numpy() - ort_out[0])))
        max_diff_d = max(max_diff_d, np.max(np.abs(pt_out[1].cpu().numpy() - ort_out[1])))

    _ok(f'Max diff continuous: {max_diff_c:.2e}')
    _ok(f'Max diff discrete:  {max_diff_d:.2e}')

    if max_diff_c < 1e-5 and max_diff_d < 0.5:
        _ok(f'✅ READY → {onnx_path} ({os.path.getsize(onnx_path)/1024:.1f} KB)')
        _inf('Copy into Assets/ML-Agents/ of your Unity project')
    else:
        _inf('⚠️  Large discrepancy — check network architecture')


  [OK]  Best checkpoint loaded (ep=20, avg=0.13)

════════════════════════════════════════════════════════════
  ONNX EXPORT
════════════════════════════════════════════════════════════


C:\Users\User\AppData\Local\Temp\ipykernel_35644\4149599749.py:42: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if self.grid_c is not None and obs_0.shape[1] == self.grid_c:


============= Diagnostic Run torch.onnx.export version 2.0.1+cu118 =============
verbose: False, log level: Level.ERROR
======================= 0 NONE 0 NOTE 0 WARNING 0 ERROR ========================

  [OK]  exported FoodCollector_REINFORCE.onnx (8267.2 KB)
  [OK]  metadata_props added

════════════════════════════════════════════════════════════
  VALIDATION
════════════════════════════════════════════════════════════
  [OK]  onnx.checker — OK
  [-->] continuous_actions shape: (1, 3),  range: [-0.1949, 0.0552]
  [-->] discrete_actions  shape: (1, 1),  values: [1]
  [OK]  onnxruntime inference — OK (single + batch)

════════════════════════════════════════════════════════════
  PYTORCH vs ONNX
════════════════════════════════════════════════════════════
  [OK]  Max diff continuous: 2.63e-07
  [OK]  Max diff discrete:  0.00e+00
  [OK]  ✅ READY → FoodCollector_REINFORCE.onnx (8267.5 KB)
  [-->] Copy into Assets/ML-Agents/ of your Unity project


## §8 · Финальная проверка выходов ONNX

In [15]:
try:
    import onnx
    m         = onnx.load(onnx_path)
    onnx.checker.check_model(m)
    out_names = [o.name for o in m.graph.output]
    required  = [
        'continuous_actions', 'discrete_actions',
        'continuous_action_output_shape', 'discrete_action_output_shape',
        'version_number', 'memory_size',
    ]
    _ok('ONNX model validated')
    for name in required:
        tag = '[OK] ' if name in out_names else '[!!] '
        print(f'    {tag}{name}')
except ImportError:
    _inf('onnx not installed — skipping validation')
except Exception as e:
    _err(f'Validation failed: {e}')

try:
    import onnxruntime as ort
    sess = ort.InferenceSession(onnx_path)

    # [FIX 8] Используем grid-форму [1, C, H, W] — не flat вектор
    outs = sess.run(None, {
        'obs_0':        np.random.randn(1, GRID_C, GRID_H, GRID_W).astype(np.float32),
        'action_masks': np.ones((1, total_discrete_mask), np.float32),
    })
    _ok('ORT inference OK')
    for name, val in zip([o.name for o in sess.get_outputs()], outs):
        if isinstance(val, np.ndarray):
            print(f'    {name}: shape={val.shape}  sample={val.flatten()[:4]}')
except ImportError:
    _inf('onnxruntime not installed — skipping inference test')
except Exception as e:
    _err(f'Inference test failed: {e}')


  [OK]  ONNX model validated
    [OK] continuous_actions
    [OK] discrete_actions
    [OK] continuous_action_output_shape
    [OK] discrete_action_output_shape
    [OK] version_number
    [OK] memory_size
  [OK]  ORT inference OK
    continuous_actions: shape=(1, 3)  sample=[-0.10200218  0.0356223  -0.10738558]
    discrete_actions: shape=(1, 1)  sample=[0]
    continuous_action_output_shape: shape=(1,)  sample=[3.]
    discrete_action_output_shape: shape=(1,)  sample=[2.]
    version_number: shape=(1,)  sample=[3.]
    memory_size: shape=(1,)  sample=[0.]


## §9 · Cleanup

In [16]:
writer.close()
env.close()

_hdr('PIPELINE COMPLETE')
print(f"""
  Следующие шаги:
  1. Скопируйте  {onnx_path}  в Assets/ Unity-проекта
  2. В Inspector агента: Model ← перетащите .onnx
  3. Behavior Type = Inference Only
  4. Play и наблюдайте!
""")



════════════════════════════════════════════════════════════
  PIPELINE COMPLETE
════════════════════════════════════════════════════════════

  Следующие шаги:
  1. Скопируйте  FoodCollector_REINFORCE.onnx  в Assets/ Unity-проекта
  2. В Inspector агента: Model ← перетащите .onnx
  3. Behavior Type = Inference Only
  4. Play и наблюдайте!



In [17]:
print('TensorBoard log_dir =', log_dir)

import time, socket
from IPython.display import IFrame, display


def _wait_port(host, port, timeout=10.0):
    start = time.time()
    while time.time() - start < timeout:
        try:
            with socket.create_connection((host, port), timeout=1):
                return True
        except Exception:
            time.sleep(0.3)
    return False


tb_url = None
try:
    from tensorboard import program
    tb = program.TensorBoard()
    tb.configure(argv=[None, '--logdir', log_dir, '--port', '6006', '--host', 'localhost'])
    tb_url = tb.launch()
    print('TensorBoard launched programmatically at', tb_url)
except Exception as e:
    print('Programmatic start failed:', e)

if tb_url is None:
    try:
        import subprocess, shlex
        cmd  = f'tensorboard --logdir "{log_dir}" --port 6006 --host localhost'
        p    = subprocess.Popen(shlex.split(cmd),
                                stdout=subprocess.DEVNULL,
                                stderr=subprocess.DEVNULL)
        print('Started tensorboard subprocess, pid=', p.pid)
    except Exception as e2:
        print('Failed to spawn subprocess:', e2)
        print(f'  tensorboard --logdir "{log_dir}" --port 6006 --host localhost')

    if _wait_port('localhost', 6006, timeout=15.0):
        tb_url = 'http://localhost:6006/'
        print('TensorBoard is available at', tb_url)
    else:
        print('Timed out waiting for TensorBoard on port 6006')

if tb_url is not None:
    display(IFrame(tb_url, width=1000, height=600))
else:
    print('TensorBoard не запущен. Откройте URL вручную после старта.')


TensorBoard log_dir = runs\FC_REINFORCE_1773659284
TensorBoard launched programmatically at http://localhost:6006/
